# Batch Raw NP Waveform Traces

This notebook uses the Reach15 `.env` session-loading flow to find the selected BombCell run, batch plot raw waveform traces for every selected unit across all selected probes, and save the plots under that BombCell root.

Output folders:
- `raw_mean/`
- `raw_overlay/`

## Imports

In [2]:
from pathlib import Path
import sys
import pandas as pd

reach15_dir = Path().resolve().parent
if str(reach15_dir) not in sys.path:
    sys.path.insert(0, str(reach15_dir))

from helper_func.export_bc_classification_reasons import (
    collect_probe_dirs,
    export_all_probes,
    resolve_session_paths,
)

from helper_func.raw_waveform_batch import (
    collect_probe_contexts,
    render_raw_waveform_batch,
    resolve_session_waveform_context,
)

print('Notebook directory:', Path().resolve())
print('Reach15 root added to sys.path:', reach15_dir)

Notebook directory: C:\Users\user\Documents\github\bombcell\mice\Reach15\analyze_bc_results
Reach15 root added to sys.path: C:\Users\user\Documents\github\bombcell\mice\Reach15


## Settings

In [3]:
# Optional explicit .env path. Leave as None to use the default dotenv discovery behavior.
ENV_PATH = None

# 1, 2, or 3 corresponding to the .env session blocks.
SESSION_SELECTION = 3

# Use None to batch all probes found in the selected BombCell root.
# Example: ['A', 'C', 'F']
PROBES = None

# Automatic cluster selection per probe.
# Set to None to plot every unit on each selected probe.
MAX_UNITS_PER_PROBE = None

# Optional label filter if the BombCell quality metrics file has a label column.
# Examples: 'GOOD', 'MUA', ['GOOD', 'NON-SOMA']
LABEL_FILTER = None

# Optional explicit cluster list override.
# Example: {'A': [3, 15], 'B': [153]}
CLUSTERS_BY_PROBE = None

# Leave as None to write raw_mean/ and raw_overlay/ directly inside the selected BombCell root.
OUTPUT_SUBDIR = None

N_CHANNELS = 384
FS = 30000
NEIGHBOR_RADIUS = 6
N_SPIKES_TO_PLOT = 40
PRE_MS = 2.0
POST_MS = 3.0
IGNORE_EDGES_S = 1.0
SEED = 0

## Resolve Session and Preview

In [4]:
session_data, cfg, bombcell_root, raw_path_by_probe = resolve_session_waveform_context(
    env_path=ENV_PATH,
    session_selection=SESSION_SELECTION,
)
probe_contexts = collect_probe_contexts(
    bombcell_root=bombcell_root,
    raw_path_by_probe=raw_path_by_probe,
    probes=PROBES,
)

session_key_map = {
    1: ('NP_FILE', 'BOMBCELL'),
    2: ('NP_FILE_01', 'BOMBCELL_01'),
    3: ('NP_FILE_02', 'BOMBCELL_02'),
}
np_key, bc_key = session_key_map[SESSION_SELECTION]

print('Recording root:', cfg['recording_root'])
print('NP file from .env:', session_data[np_key])
print('BombCell folder from .env:', session_data[bc_key])
print('BombCell root:', bombcell_root)
print('Probes to process:', ', '.join(probe_contexts.keys()))
for probe, context in probe_contexts.items():
    print(f"  Probe {probe}: ks_dir={context['ks_dir']}")
    print(f"            bin_path={context['bin_path']}")

MOUSE loaded: Reach15
-- Behavioral Files --
BEHAVIORAL_FOLDER loaded: grant_reach15_swingDoor-christie
-- Neuropixels Sessions --
Session 1: NP_FILE=Reach15_20260129_session003_NP_Recording_2026-01-29_14-30-01 DATE=20260129 SESSION=session003 BOMBCELL=bombcell_batch_20260305_1130
Session 2: NP_FILE=Reach15_20260129_session004_NP_Recording_02_2026-01-29_16-50-32 DATE=20260129 SESSION=session004 BOMBCELL=NA
Session 3: NP_FILE=Reach15_20260201_session007_NP_Recording_Number02_2026-02-01_18-25-00 DATE=20260201 SESSION=session007 BOMBCELL=bombcell_batch_20260304_1536
-- Config Defaults --
RECORDINGS_ROOT loaded: H:/Grant/Neuropixels/Kilosort_Recordings
OPEN_EPHYS_CONTINUOUS_SUBPATH loaded: Record Node 103/experiment1/recording1/continuous
STRUCTURE_OEBIN_SUBPATH loaded: Record Node 103/experiment1/recording1/structure.oebin
NP20_PROBES loaded: A,C,D
Recording root: H:\Grant\Neuropixels\Kilosort_Recordings\Reach15_20260201_session007_NP_Recording_Number02_2026-02-01_18-25-00
NP file from .e

## Run Batch Plot Export

In [5]:
summary_df, output_root = render_raw_waveform_batch(
    bombcell_root=bombcell_root,
    raw_path_by_probe=raw_path_by_probe,
    probes=list(probe_contexts.keys()),
    clusters_by_probe=CLUSTERS_BY_PROBE,
    max_units_per_probe=MAX_UNITS_PER_PROBE,
    label_filter=LABEL_FILTER,
    output_subdir=OUTPUT_SUBDIR,
    n_channels=N_CHANNELS,
    fs=FS,
    neighbor_radius=NEIGHBOR_RADIUS,
    n_spikes_to_plot=N_SPIKES_TO_PLOT,
    pre_ms=PRE_MS,
    post_ms=POST_MS,
    seed=SEED,
    ignore_edges_s=IGNORE_EDGES_S,
)

print('Saved output root:', output_root)
display(summary_df)

KeyboardInterrupt: 

## Quick Summary

In [ ]:
summary_counts = summary_df['status'].value_counts(dropna=False).rename_axis('status').reset_index(name='count')
display(summary_counts)

ok_df = summary_df.loc[summary_df['status'] == 'OK'].copy()
if not ok_df.empty:
    display(ok_df[['probe', 'cluster_id', 'center_chan', 'center_channel_source', 'overlay_path', 'mean_path']])

summary_csv = Path(output_root) / 'raw_waveform_plot_summary.csv'
print('Summary CSV:', summary_csv)